### 신뢰 신호 통계검증

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import chi2_contingency
import os
import warnings

warnings.filterwarnings('ignore')

# 1. 데이터 로드 및 전처리
base_path = '../../data/preprocessed/'
df_graded = pd.read_csv(os.path.join(base_path, 'steam_indie_games_graded.csv'))
df_silence = pd.read_csv(os.path.join(base_path, 'steam_indie_games_silence.csv'))

df = pd.concat([df_graded, df_silence], ignore_index=True)
df.drop_duplicates(subset=['appid'], inplace=True)

# 1단계(1~9개), 2단계(10~49개)만 추출
df = df[df['total_reviews'] > 0].copy()
df['stage'] = df['total_reviews'].apply(lambda x: 1 if 1 <= x <= 9 else (2 if 10 <= x <= 49 else 3))
df_target = df[df['stage'].isin([1, 2])].copy()

# '성공(Promotion)' = 2단계 진입
df_target['is_promoted'] = (df_target['stage'] == 2).astype(int)

# ====================================================================
# [데이터 정밀 추출] categories와 기존 컬럼에서 신뢰 신호 Parsing
# =====================================================================
df_target['categories'] = df_target['categories'].fillna('')

# 1. 도전과제: categories 텍스트에 있거나, 실제 도전과제 수가 0 이상인 경우
df_target['has_achievements'] = df_target['categories'].str.contains('Achievement|도전과제', case=False) | (df_target['achievements_total'] > 0)
# 2. 컨트롤러: Full, Partial 모두 포함
df_target['has_controller'] = df_target['categories'].str.contains('Controller|컨트롤러', case=False)
# 3. 스팀 클라우드
df_target['has_cloud'] = df_target['categories'].str.contains('Cloud|클라우드', case=False)
# 4. 가족 공유
df_target['has_family_sharing'] = df_target['categories'].str.contains('Family Sharing|가족 공유', case=False)
# 5. 싱글 플레이
df_target['is_single_player'] = df_target['categories'].str.contains('Single-player|Singleplayer|싱글 플레이', case=False)

# 6. 자체 퍼블리싱 여부 (개발사 == 퍼블리셔)
df_target['is_self_published'] = (df_target['developers'] == df_target['publishers'])

# 7. Mac, Linux 지원 여부 (기존 컬럼 사용)
df_target['mac'] = df_target['mac'].fillna(False).astype(bool)
df_target['linux'] = df_target['linux'].fillna(False).astype(bool)

# =====================================================================
# [검증 파트] 데이터 타당성 및 통계적 유의성(Chi-Square) 확인
# =====================================================================
trust_features = {
    "도전과제 (Achievements)": "has_achievements",
    "클라우드 (Steam Cloud)": "has_cloud",
    "컨트롤러 지원 (Controller)": "has_controller",
    "Mac 지원": "mac",
    "Linux 지원": "linux",
    "싱글 플레이 (Single Player)": "is_single_player",
    "1인/자체개발 (Self-Published)": "is_self_published",
    "가족 공유 (Family Sharing)": "has_family_sharing"
}

validation_results = []

for label, col in trust_features.items():
    df_target[col] = df_target[col].fillna(False).astype(bool)
    
    count_true = df_target[col].sum()
    count_false = len(df_target) - count_true
    support_rate = (count_true / len(df_target)) * 100
    
    contingency_table = pd.crosstab(df_target[col], df_target['is_promoted'])
    
    if contingency_table.shape == (2, 2): 
        chi2, p_value, dof, expected = chi2_contingency(contingency_table)
        
        success_rate_with = df_target[df_target[col] == True]['is_promoted'].mean() * 100
        success_rate_without = df_target[df_target[col] == False]['is_promoted'].mean() * 100
        gap = success_rate_with - success_rate_without # 지원할 때 성공률이 얼마나 오르는가?
    else:
        p_value, success_rate_with, success_rate_without, gap = np.nan, np.nan, np.nan, np.nan
        
    validation_results.append({
        "Feature": label,
        "True_Count (지원)": count_true,
        "False_Count (미지원)": count_false,
        "Support_Rate (%)": support_rate,
        "Success_Gap (%p)": gap,
        "P-Value": p_value,
        "Is_Significant (p<0.05)": "✅ Yes" if p_value < 0.05 else "❌ No"
    })

val_df = pd.DataFrame(validation_results).sort_values(by="Success_Gap (%p)", ascending=False)

print("\n" + "="*90)
print("[데이터 타당성 증명 리포트] 신뢰 신호(Trust Features) 분석 전 검증")
print("="*90)
# p-value는 소수점 4자리까지 표기하여 확실한 유의성 어필
print(val_df.round({'Support_Rate (%)': 1, 'Success_Gap (%p)': 2, 'P-Value': 4}).to_string(index=False))
print("="*90)

# =====================================================================
# [시각화] P-Value 통과한 지표들의 1단계 탈출 기여도 (막대 그래프)
# =====================================================================
# 유의미한 데이터만 필터링
valid_features = val_df[val_df['Is_Significant (p<0.05)'] == '✅ Yes'].copy()

fig = px.bar(
    valid_features.sort_values("Success_Gap (%p)"),
    x="Success_Gap (%p)",
    y="Feature",
    orientation="h",
    color="Success_Gap (%p)",
    color_continuous_scale="RdBu",
    title="신뢰 신호가 '1단계 -> 2단계' 진입 확률에 미치는 영향 (통계적 유의성 검증 완료)",
    text_auto='.1f'
)
fig.update_layout(
    xaxis_title="2단계 진입 확률 차이 (지원 시 상승치 %p)", 
    yaxis_title="신뢰 신호",
    plot_bgcolor='white'
)
fig.show()


[데이터 타당성 증명 리포트] 신뢰 신호(Trust Features) 분석 전 검증
                 Feature  True_Count (지원)  False_Count (미지원)  Support_Rate (%)  Success_Gap (%p)  P-Value Is_Significant (p<0.05)
     도전과제 (Achievements)             7165               4661              60.6             16.30   0.0000                   ✅ Yes
      클라우드 (Steam Cloud)             2928               8898              24.8             15.81   0.0000                   ✅ Yes
    컨트롤러 지원 (Controller)             4586               7240              38.8              9.09   0.0000                   ✅ Yes
                  Mac 지원             1583              10243              13.4              8.42   0.0000                   ✅ Yes
                Linux 지원             1415              10411              12.0              6.22   0.0000                   ✅ Yes
  싱글 플레이 (Single Player)            11469                357              97.0             -6.84   0.0123                   ✅ Yes
1인/자체개발 (Self-Published)             9313 

크기효과 추가

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import chi2_contingency
import os
import warnings

warnings.filterwarnings('ignore')

# 1. 데이터 로드 및 전처리 (기존과 동일)
base_path = '../../data/preprocessed/'
df_graded = pd.read_csv(os.path.join(base_path, 'steam_indie_games_graded.csv'))
df_silence = pd.read_csv(os.path.join(base_path, 'steam_indie_games_silence.csv'))

df = pd.concat([df_graded, df_silence], ignore_index=True)
df.drop_duplicates(subset=['appid'], inplace=True)

df = df[df['total_reviews'] > 0].copy()
df['stage'] = df['total_reviews'].apply(lambda x: 1 if 1 <= x <= 9 else (2 if 10 <= x <= 49 else 3))
df_target = df[df['stage'].isin([1, 2])].copy()
df_target['is_promoted'] = (df_target['stage'] == 2).astype(int)

# 신뢰 신호 추출
df_target['categories'] = df_target['categories'].fillna('')
df_target['has_achievements'] = df_target['categories'].str.contains('Achievement|도전과제', case=False) | (df_target['achievements_total'] > 0)
df_target['has_controller'] = df_target['categories'].str.contains('Controller|컨트롤러', case=False)
df_target['has_cloud'] = df_target['categories'].str.contains('Cloud|클라우드', case=False)
df_target['has_family_sharing'] = df_target['categories'].str.contains('Family Sharing|가족 공유', case=False)
df_target['is_single_player'] = df_target['categories'].str.contains('Single-player|Singleplayer|싱글 플레이', case=False)
df_target['is_self_published'] = (df_target['developers'] == df_target['publishers'])
df_target['mac'] = df_target['mac'].fillna(False).astype(bool)
df_target['linux'] = df_target['linux'].fillna(False).astype(bool)

# 2. 효과 크기를 포함한 검증 로직
trust_features = {
    "도전과제 (Achievements)": "has_achievements",
    "클라우드 (Steam Cloud)": "has_cloud",
    "컨트롤러 지원 (Controller)": "has_controller",
    "Mac 지원": "mac",
    "Linux 지원": "linux",
    "싱글 플레이 (Single Player)": "is_single_player",
    "1인/자체개발 (Self-Published)": "is_self_published",
    "가족 공유 (Family Sharing)": "has_family_sharing"
}

validation_results = []

for label, col in trust_features.items():
    df_target[col] = df_target[col].fillna(False).astype(bool)
    
    count_true = df_target[col].sum()
    support_rate = (count_true / len(df_target)) * 100
    
    # 교차표 생성
    contingency_table = pd.crosstab(df_target[col], df_target['is_promoted'])
    
    if contingency_table.shape == (2, 2): 
        chi2, p_value, dof, expected = chi2_contingency(contingency_table)
        
        success_rate_with = df_target[df_target[col] == True]['is_promoted'].mean() * 100
        success_rate_without = df_target[df_target[col] == False]['is_promoted'].mean() * 100
        gap = success_rate_with - success_rate_without
        
        # [추가 1] 효과 크기 (Cramér's V)
        # 0.1(작음), 0.3(중간), 0.5(큼). 범주형 데이터 상관관계의 강도.
        n = contingency_table.sum().sum()
        cramers_v = np.sqrt(chi2 / n)
        
        # [추가 2] 오즈비 (Odds Ratio)
        # 해당 기능을 지원할 때 2단계 진입 확률이 '몇 배' 더 높은가?
        try:
            A = contingency_table.loc[False, 0] # 미지원 & 실패
            B = contingency_table.loc[False, 1] # 미지원 & 성공
            C = contingency_table.loc[True, 0]  # 지원 & 실패
            D = contingency_table.loc[True, 1]  # 지원 & 성공
            odds_ratio = (D * A) / (C * B)
        except KeyError:
            odds_ratio = np.nan
            
    else:
        p_value, gap, cramers_v, odds_ratio = np.nan, np.nan, np.nan, np.nan
        
    validation_results.append({
        "Feature": label,
        "Support_Rate (%)": support_rate,
        "Success_Gap (%p)": gap,
        "P-Value": p_value,
        "Cramér's V (상관강도)": cramers_v,
        "Odds Ratio (성공배수)": odds_ratio,
        "Is_Significant": "✅" if p_value < 0.05 else "❌"
    })

val_df = pd.DataFrame(validation_results).sort_values(by="Odds Ratio (성공배수)", ascending=False)

print("\n" + "="*100)
print("[데이터 타당성 증명 리포트] 효과 크기(Effect Size) 및 오즈비(Odds Ratio) 분석")
print("="*100)
print(val_df.round({'Support_Rate (%)': 1, 'Success_Gap (%p)': 2, 'P-Value': 4, "Cramér's V (상관강도)": 3, "Odds Ratio (성공배수)": 2}).to_string(index=False))
print("="*100)


[데이터 타당성 증명 리포트] 효과 크기(Effect Size) 및 오즈비(Odds Ratio) 분석
                 Feature  Support_Rate (%)  Success_Gap (%p)  P-Value  Cramér's V (상관강도)  Odds Ratio (성공배수) Is_Significant
     도전과제 (Achievements)              60.9             17.75   0.0000              0.175               2.11              ✅
      클라우드 (Steam Cloud)              24.9             16.91   0.0000              0.148               1.99              ✅
  가족 공유 (Family Sharing)              99.3             12.63   0.0306              0.020               1.73              ✅
    컨트롤러 지원 (Controller)              39.2             10.53   0.0000              0.104               1.54              ✅
                  Mac 지원              13.6              9.46   0.0000              0.065               1.47              ✅
                Linux 지원              12.0              6.82   0.0000              0.045               1.32              ✅
  싱글 플레이 (Single Player)              97.6              5.95   0.0560            